In [ ]:
# ============================================================
# CELL 1 - Imports
# ============================================================

import os
import pickle
import random
import numpy as np
import pandas as pd
import torch
import networkx as nx

from tqdm import tqdm

from torch_geometric.nn import GCNConv
from torch_geometric.nn.models import DeepGraphInfomax
from torch_geometric.utils import from_networkx
from torch_geometric.loader import DataLoader

In [ ]:
# ============================================================
# CELL 2 - Reproducibility and device
# ============================================================

def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ============================================================
# CELL 3 - Paths
# ============================================================

graphs_path = "village51_link_prediction/graphs_adamic_adar_nsum.pkl"

# ALL-FEATURES matrix
features_path = "features_aligned_village51.csv"

# CIRCLES-ONLY input: ARD matrix
ard_path = "ard_matrix_village51.csv"

original_edges_path = "village51.edges"

output_folder = "village51_dgi"
os.makedirs(output_folder, exist_ok=True)

In [ ]:
# ============================================================
# CELL 4 - Load synthetic graphs
# ============================================================

def load_augmented_graphs(graphs_path):
    with open(graphs_path, "rb") as f:
        graphs = pickle.load(f)

    return graphs


graphs = load_augmented_graphs(graphs_path)

print("Number of synthetic graphs:", len(graphs))
print("Example synthetic graph nodes:", graphs[0].number_of_nodes())
print("Example synthetic graph edges:", graphs[0].number_of_edges())
print("First graph nodes:", list(graphs[0].nodes())[:10])

In [ ]:
# ============================================================
# CELL 5 - Load ALL-FEATURES, ARD CIRCLES-ONLY,
#          and graph-structure-only identity matrices
# ============================================================

def load_matrix_for_synthetic_order(
    matrix_path,
    reference_node_ids=None,
    matrix_name="matrix"
):
    """
    Loads a node-level matrix and aligns it to the positional node order
    used by synthetic graphs: 1..n.

    ARD defines the canonical original-node order because synthetic graphs
    were generated from the ARD matrix.
    """

    matrix_df_original = pd.read_csv(
        matrix_path,
        index_col=0
    )

    matrix_df_original.index = matrix_df_original.index.astype(int)
    matrix_df_original.index.name = "node_id"

    if reference_node_ids is not None:
        reference_node_ids = [
            int(node_id)
            for node_id in reference_node_ids
        ]

        missing_nodes = (
            set(reference_node_ids)
            - set(matrix_df_original.index)
        )

        extra_nodes = (
            set(matrix_df_original.index)
            - set(reference_node_ids)
        )

        if missing_nodes:
            raise ValueError(
                f"{matrix_name}: missing nodes relative to ARD reference: "
                f"{sorted(missing_nodes)[:10]}"
            )

        if extra_nodes:
            raise ValueError(
                f"{matrix_name}: extra nodes not present in ARD reference: "
                f"{sorted(extra_nodes)[:10]}"
            )

        matrix_df_original = matrix_df_original.loc[
            reference_node_ids
        ]

    original_node_ids = matrix_df_original.index.tolist()

    # Synthetic graphs use positional IDs 1..n
    matrix_df = matrix_df_original.reset_index(drop=True)
    matrix_df.index = range(1, len(matrix_df) + 1)
    matrix_df.index.name = "node_id"

    matrix_tensor = torch.tensor(
        matrix_df.to_numpy(),
        dtype=torch.float32
    )

    return (
        matrix_df_original,
        matrix_df,
        matrix_tensor,
        original_node_ids
    )


# ------------------------------------------------------------
# ARD CIRCLES-ONLY MATRIX
# This defines the canonical node order.
# ------------------------------------------------------------

(
    ard_df_original,
    ard_df,
    ard_tensor,
    original_node_ids
) = load_matrix_for_synthetic_order(
    matrix_path=ard_path,
    reference_node_ids=None,
    matrix_name="ARD"
)

print("=== ARD CIRCLES-ONLY MATRIX ===")
print("ARD shape:", ard_tensor.shape)
print("ARD columns:", list(ard_df.columns))
print("First positional node IDs:", list(ard_df.index)[:10])
print("First original node IDs:", original_node_ids[:10])

display(ard_df.head())


# ------------------------------------------------------------
# ALL-FEATURES MATRIX
# Explicitly reordered to match ARD node order.
# ------------------------------------------------------------

(
    features_df_original,
    features_df,
    features_tensor,
    features_original_node_ids
) = load_matrix_for_synthetic_order(
    matrix_path=features_path,
    reference_node_ids=original_node_ids,
    matrix_name="ALL-FEATURES"
)

assert features_original_node_ids == original_node_ids, (
    "Features are not aligned with ARD node order."
)

print("\n=== ALL-FEATURES MATRIX ===")
print("Features shape:", features_tensor.shape)
print("First feature index:", list(features_df.index)[:10])
print("First original node IDs:", features_original_node_ids[:10])

display(features_df.head())


# ------------------------------------------------------------
# GRAPH-STRUCTURE-ONLY INPUT
#
# Identity matrix:
# no demographic attributes, no ARD and no membership information.
#
# Each node receives only a unique technical identifier so that
# DGI can learn from the graph connections.
# ------------------------------------------------------------

n_nodes = ard_tensor.shape[0]

structure_only_df = pd.DataFrame(
    np.eye(n_nodes, dtype=np.float32),
    index=range(1, n_nodes + 1),
    columns=[
        f"node_identity_{node_id}"
        for node_id in range(1, n_nodes + 1)
    ]
)

structure_only_df.index.name = "node_id"

structure_only_tensor = torch.tensor(
    structure_only_df.to_numpy(),
    dtype=torch.float32
)

print("\n=== GRAPH-STRUCTURE-ONLY IDENTITY MATRIX ===")
print("Identity matrix shape:", structure_only_tensor.shape)

print(
    "Every node has exactly one active identity feature:",
    (structure_only_df.sum(axis=1) == 1).all()
)

display(structure_only_df.iloc[:5, :10])


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

# Circle variables should NOT be in all-features.
# "subcaste" is allowed because it is not one of the circle variables.

for forbidden_prefix in ["caste_", "resp_gend_"]:

    matched_cols = [
        col
        for col in features_df.columns
        if col.startswith(forbidden_prefix)
    ]

    print(f"{forbidden_prefix} columns in all-features:", matched_cols)

    assert len(matched_cols) == 0, (
        f"{forbidden_prefix} should not be in all-features matrix."
    )

print("\nALL-FEATURES matrix correctly excludes circle variables.")

subcaste_cols = [
    col
    for col in features_df.columns
    if col.startswith("subcaste_")
]

print("\nsubcaste columns present:", len(subcaste_cols))

assert ard_tensor.shape[0] == features_tensor.shape[0], (
    "ARD and all-features must have the same number of nodes."
)

assert ard_tensor.shape[0] == structure_only_tensor.shape[0], (
    "ARD and identity matrix must have the same number of nodes."
)

assert structure_only_tensor.shape[1] == n_nodes, (
    "Identity matrix must have one feature per node."
)

assert (structure_only_df.sum(axis=1) == 1).all(), (
    "Each node must have exactly one active identity feature."
)

assert (ard_df.to_numpy() >= 0).all(), (
    "ARD cannot contain negative values."
)

print("\nARD matrix correctly loaded as circles-only input.")
print("Identity matrix correctly loaded as graph-structure-only input.")

In [ ]:
# ============================================================
# CELL 6 - Alignment checks
# ============================================================

def check_graph_feature_alignment(G, features_df, features_tensor):
    print("Graph nodes:", G.number_of_nodes())
    print("Features rows:", features_tensor.shape[0])
    print("First graph nodes:", list(G.nodes())[:10])
    print("First feature index:", list(features_df.index)[:10])

    assert G.number_of_nodes() == features_tensor.shape[0], (
        "Mismatch: graph nodes and feature rows differ."
    )

    assert list(G.nodes()) == list(features_df.index), (
        "Mismatch: graph node order and feature row order differ."
    )

    print("Graph-feature alignment OK.")


def check_all_graphs_alignment(graphs, features_df, features_tensor):
    for i, G in enumerate(graphs, start=1):
        assert G.number_of_nodes() == features_tensor.shape[0], (
            f"Graph {i}: number of nodes differs from features."
        )

        assert list(G.nodes()) == list(features_df.index), (
            f"Graph {i}: node order differs from feature order."
        )

    print(
        f"All {len(graphs)} synthetic graphs are aligned with features."
    )


print("\n=== Alignment check: all-features ===")
check_graph_feature_alignment(
    G=graphs[0],
    features_df=features_df,
    features_tensor=features_tensor
)

check_all_graphs_alignment(
    graphs=graphs,
    features_df=features_df,
    features_tensor=features_tensor
)

print("\n=== Alignment check: ARD circles-only ===")
check_graph_feature_alignment(
    G=graphs[0],
    features_df=ard_df,
    features_tensor=ard_tensor
)

check_all_graphs_alignment(
    graphs=graphs,
    features_df=ard_df,
    features_tensor=ard_tensor
)

print("\n=== Alignment check: graph-structure-only identity ===")
check_graph_feature_alignment(
    G=graphs[0],
    features_df=structure_only_df,
    features_tensor=structure_only_tensor
)

check_all_graphs_alignment(
    graphs=graphs,
    features_df=structure_only_df,
    features_tensor=structure_only_tensor
)

In [ ]:
# ============================================================
# CELL 7 - Convert NetworkX graphs to PyG
# ============================================================

def graph_to_pyg_data(G, features_tensor):
    if G.number_of_nodes() != features_tensor.shape[0]:
        raise ValueError(
            f"Mismatch: graph has {G.number_of_nodes()} nodes, "
            f"features have {features_tensor.shape[0]} rows."
        )

    data = from_networkx(G)
    data.x = features_tensor

    return data


def graphs_to_pyg_data_list(graphs, features_tensor):
    data_list = []

    for G in tqdm(graphs, desc="Converting graphs to PyG"):
        data = graph_to_pyg_data(G, features_tensor)
        data_list.append(data)

    return data_list

In [ ]:
# ============================================================
# CELL 8 - GCN Encoder and DGI model
# ============================================================

class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = GCNConv(in_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        return self.conv(x, edge_index).relu()


def corruption(x, edge_index, batch=None):
    perm = torch.randperm(x.size(0), device=x.device)
    return x[perm], edge_index, batch


def build_dgi_model(num_features, embedding_dim):
    encoder = GCNEncoder(
        in_channels=num_features,
        out_channels=embedding_dim
    )

    model = DeepGraphInfomax(
        hidden_channels=embedding_dim,
        encoder=encoder,
        summary=lambda z, *args, **kwargs: torch.sigmoid(z.mean(dim=0)),
        corruption=corruption,
    )

    return model

In [ ]:
# ============================================================
# CELL 9 - Save utilities
# ============================================================

def save_embeddings(embeddings, output_path):
    np.save(output_path, embeddings)


def save_loss_histories(loss_histories, output_path):
    with open(output_path, "wb") as f:
        pickle.dump(loss_histories, f)


def save_model_state(model, output_path):
    torch.save(model.state_dict(), output_path)

In [ ]:
# ============================================================
# CELL 10 - METHOD 1
# Independent DGI per synthetic graph + mean
# ============================================================

def train_dgi_single_graph(
    G,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123,
    verbose=True,
):
    if seed is not None:
        set_seed(seed)

    device = get_device()

    data = graph_to_pyg_data(G, features_tensor).to(device)

    model = build_dgi_model(
        num_features=data.num_features,
        embedding_dim=embedding_dim
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_loss = float("inf")
    best_training_embeddings = None
    best_training_epoch = None
    patience_counter = 0
    loss_history = []

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()

        pos_z, neg_z, summary = model(data.x, data.edge_index)
        loss = model.loss(pos_z, neg_z, summary)

        loss_value = loss.item()
        loss_history.append(loss_value)

        if loss_value < best_loss:
            best_loss = loss_value
            best_training_epoch = epoch
            # Positive node embeddings produced directly during
            # training, before the optimizer update.
            best_training_embeddings = (
                pos_z.detach().cpu().numpy().copy()
            )
            patience_counter = 0
        else:
            patience_counter += 1

        loss.backward()
        optimizer.step()

        if verbose and epoch % 50 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss_value:.4f}")

        if patience_counter >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break

    model.eval()

    with torch.no_grad():
        embeddings = model.encoder(
            data.x,
            data.edge_index
        ).cpu().numpy()

    if best_training_embeddings is None:
        raise RuntimeError(
            "No training-time embeddings were retained for the graph."
        )

    return (
        embeddings,
        loss_history,
        best_training_embeddings,
        best_training_epoch,
        best_loss,
    )


def train_dgi_ensemble_mean_independent(
    graphs,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123,
    verbose=False,
):
    set_seed(seed)

    if len(graphs) == 0:
        raise ValueError("No graphs provided.")

    n_nodes = features_tensor.shape[0]
    embeddings_sum = np.zeros((n_nodes, embedding_dim), dtype=np.float32)
    all_loss_histories = []

    for i, G in enumerate(tqdm(graphs, desc="Independent DGI ensemble"), start=1):
        if verbose:
            print(f"\n=== Graph {i}/{len(graphs)} ===")

        (
            embeddings,
            loss_history,
            _,
            _,
            _,
        ) = train_dgi_single_graph(
            G=G,
            features_tensor=features_tensor,
            embedding_dim=embedding_dim,
            lr=lr,
            max_epochs=max_epochs,
            patience=patience,
            seed=None,
            verbose=verbose,
        )

        embeddings_sum += embeddings
        all_loss_histories.append(loss_history)

    embeddings_mean = embeddings_sum / len(graphs)

    return embeddings_mean, all_loss_histories

In [ ]:
# ============================================================
# CELL 11 - METHOD 2
# Shared sequential DGI across synthetic graphs
#
# ADDITIONAL EXPERIMENT:
# retain embeddings produced directly DURING training.
# ============================================================

def train_dgi_shared_across_graphs(
    graphs,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123,
    verbose=True,
):
    """
    One Encoder methodology.

    Outputs:
    1) post_training_embeddings_mean:
       after training, the shared encoder is applied again to all
       reconstructed graphs and embeddings are averaged by node.

    2) training_embeddings_mean:
       embeddings produced directly during the epoch with the
       lowest mean training loss are averaged by node, without
       re-applying the encoder after training.

    Important:
    in One Encoder, the model is updated after each graph, so the
    training-time embeddings within one epoch come from successive
    states of the same shared encoder.
    """
    set_seed(seed)

    if len(graphs) == 0:
        raise ValueError("No graphs provided.")

    device = get_device()

    data_list = graphs_to_pyg_data_list(
        graphs=graphs,
        features_tensor=features_tensor
    )

    data_list = [data.to(device) for data in data_list]

    num_features = data_list[0].num_features

    model = build_dgi_model(
        num_features=num_features,
        embedding_dim=embedding_dim
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_loss = float("inf")
    best_training_embeddings_mean = None
    best_training_epoch = None

    patience_counter = 0
    epoch_loss_history = []

    for epoch in range(max_epochs):
        model.train()

        graph_indices = np.random.permutation(len(data_list))
        epoch_losses = []

        # Store embeddings by original graph index so that graph
        # shuffling does not affect bookkeeping.
        epoch_training_embeddings = [None] * len(data_list)

        for idx in graph_indices:
            data = data_list[idx]

            optimizer.zero_grad()

            pos_z, neg_z, summary = model(
                data.x,
                data.edge_index
            )

            # Embeddings produced directly during training,
            # before optimizer.step() changes the encoder.
            epoch_training_embeddings[idx] = (
                pos_z.detach().cpu().numpy().copy()
            )

            loss = model.loss(
                pos_z,
                neg_z,
                summary
            )

            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        epoch_mean_loss = float(np.mean(epoch_losses))
        epoch_loss_history.append(epoch_mean_loss)

        if epoch_mean_loss < best_loss:
            best_loss = epoch_mean_loss
            best_training_epoch = epoch
            patience_counter = 0

            if any(
                emb is None
                for emb in epoch_training_embeddings
            ):
                raise RuntimeError(
                    "Missing training-time embeddings for at least one graph."
                )

            best_training_embeddings_mean = np.mean(
                np.stack(
                    epoch_training_embeddings,
                    axis=0
                ),
                axis=0
            ).astype(np.float32)

        else:
            patience_counter += 1

        if verbose and epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Mean loss across graphs: {epoch_mean_loss:.4f}"
            )

        if patience_counter >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break

    if best_training_embeddings_mean is None:
        raise RuntimeError(
            "No One Encoder training-time embeddings were retained."
        )

    # --------------------------------------------------------
    # ORIGINAL / POST-TRAINING OUTPUT
    # Re-apply the trained encoder to each reconstructed graph.
    # --------------------------------------------------------
    model.eval()

    n_nodes = features_tensor.shape[0]
    embeddings_sum = np.zeros(
        (n_nodes, embedding_dim),
        dtype=np.float32
    )
    all_graph_embeddings = []

    with torch.no_grad():
        for data in tqdm(
            data_list,
            desc="Extracting shared-DGI post-training embeddings"
        ):
            emb = model.encoder(
                data.x,
                data.edge_index
            ).cpu().numpy()

            embeddings_sum += emb
            all_graph_embeddings.append(emb)

    post_training_embeddings_mean = (
        embeddings_sum / len(data_list)
    )

    return (
        post_training_embeddings_mean,
        all_graph_embeddings,
        epoch_loss_history,
        model,
        best_training_embeddings_mean,
        best_training_epoch,
        best_loss,
    )


In [ ]:
# ============================================================
# CELL 12 - METHOD 3
# Batch DGI across synthetic graphs
#
# ADDITIONAL EXPERIMENT:
# retain embeddings produced directly DURING training.
# ============================================================

def _split_pyg_batch_embeddings(
    positive_embeddings,
    batch_vector,
    n_nodes
):
    """
    Split a PyG batch into one embedding matrix per graph.

    All reconstructed graphs contain the same number of nodes.
    """
    graph_embeddings = []

    graph_ids = torch.unique(
        batch_vector,
        sorted=True
    )

    for graph_id in graph_ids:
        mask = (
            batch_vector == graph_id
        )

        emb_graph = positive_embeddings[mask]

        if emb_graph.shape[0] != n_nodes:
            raise ValueError(
                "A graph in the batch does not contain the expected "
                "number of nodes."
            )

        graph_embeddings.append(
            emb_graph.detach().cpu().numpy().copy()
        )

    return graph_embeddings


def train_dgi_batch_across_graphs(
    graphs,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123,
    batch_size=None,
    verbose=True,
):
    """
    Batch methodology.

    The original Indian configuration is preserved:
    batch_size=None means that all reconstructed graphs are placed
    in one full batch.

    Outputs:
    1) post_training_embeddings_mean:
       after training, the shared encoder is applied again to every
       reconstructed graph and embeddings are averaged by node.

    2) training_embeddings_mean:
       embeddings produced directly during the epoch with the lowest
       training loss are averaged by node across graph realizations,
       without re-applying the encoder after training.
    """
    set_seed(seed)

    if len(graphs) == 0:
        raise ValueError("No graphs provided.")

    device = get_device()

    data_list = graphs_to_pyg_data_list(
        graphs=graphs,
        features_tensor=features_tensor
    )

    if batch_size is None:
        batch_size = len(data_list)

    loader = DataLoader(
        data_list,
        batch_size=batch_size,
        shuffle=True
    )

    num_features = data_list[0].num_features
    n_nodes = features_tensor.shape[0]

    model = build_dgi_model(
        num_features=num_features,
        embedding_dim=embedding_dim
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_loss = float("inf")
    best_training_embeddings_mean = None
    best_training_epoch = None

    patience_counter = 0
    epoch_loss_history = []

    for epoch in range(max_epochs):
        model.train()
        batch_losses = []

        epoch_training_graph_embeddings = []

        for batch in loader:
            batch = batch.to(device)

            optimizer.zero_grad()

            pos_z, neg_z, summary = model(
                batch.x,
                batch.edge_index,
                batch.batch
            )

            # Save embeddings produced directly in the training
            # forward pass, before optimizer.step().
            epoch_training_graph_embeddings.extend(
                _split_pyg_batch_embeddings(
                    positive_embeddings=pos_z,
                    batch_vector=batch.batch,
                    n_nodes=n_nodes
                )
            )

            loss = model.loss(
                pos_z,
                neg_z,
                summary
            )

            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

        epoch_mean_loss = float(np.mean(batch_losses))
        epoch_loss_history.append(epoch_mean_loss)

        if epoch_mean_loss < best_loss:
            best_loss = epoch_mean_loss
            best_training_epoch = epoch
            patience_counter = 0

            if (
                len(epoch_training_graph_embeddings)
                != len(data_list)
            ):
                raise RuntimeError(
                    "The epoch did not produce training-time embeddings "
                    "for every reconstructed graph."
                )

            best_training_embeddings_mean = np.mean(
                np.stack(
                    epoch_training_graph_embeddings,
                    axis=0
                ),
                axis=0
            ).astype(np.float32)

        else:
            patience_counter += 1

        if verbose and epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Mean batch loss: {epoch_mean_loss:.4f}"
            )

        if patience_counter >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break

    if best_training_embeddings_mean is None:
        raise RuntimeError(
            "No Batch training-time embeddings were retained."
        )

    # --------------------------------------------------------
    # ORIGINAL / POST-TRAINING OUTPUT
    # Re-apply the trained encoder to every reconstructed graph.
    # --------------------------------------------------------
    model.eval()

    embeddings_sum = np.zeros(
        (n_nodes, embedding_dim),
        dtype=np.float32
    )
    all_graph_embeddings = []

    with torch.no_grad():
        for data in tqdm(
            data_list,
            desc="Extracting batch-DGI post-training embeddings"
        ):
            data = data.to(device)

            emb = model.encoder(
                data.x,
                data.edge_index
            ).cpu().numpy()

            embeddings_sum += emb
            all_graph_embeddings.append(emb)

    post_training_embeddings_mean = (
        embeddings_sum / len(data_list)
    )

    return (
        post_training_embeddings_mean,
        all_graph_embeddings,
        epoch_loss_history,
        model,
        best_training_embeddings_mean,
        best_training_epoch,
        best_loss,
    )


In [ ]:
# ============================================================
# CELL 13 - Load and reindex original graph
# ============================================================

def load_original_graph_reindexed_from_node_order(
    edges_path,
    original_node_ids
):
    edges_df = pd.read_csv(
        edges_path,
        sep=" ",
        header=None,
        names=["from", "to"]
    )

    original_node_ids = [
        int(node_id)
        for node_id in original_node_ids
    ]

    # Map original village IDs to positional synthetic-graph IDs: 1..n
    mapping = {
        old_node: new_node
        for new_node, old_node in enumerate(
            original_node_ids,
            start=1
        )
    }

    G_original = nx.Graph()

    # Preserve all nodes, including isolates
    G_original.add_nodes_from(
        range(1, len(original_node_ids) + 1)
    )

    edges = []

    for u, v in edges_df[["from", "to"]].to_numpy():
        u = int(u)
        v = int(v)

        if u in mapping and v in mapping:
            edges.append((mapping[u], mapping[v]))

    # Remove duplicate undirected edges
    edges_set = set(
        tuple(sorted(edge))
        for edge in edges
    )

    G_original.add_edges_from(edges_set)

    return G_original


G_original = load_original_graph_reindexed_from_node_order(
    edges_path=original_edges_path,
    original_node_ids=original_node_ids
)

print("Original graph nodes:", G_original.number_of_nodes())
print("Original graph edges:", G_original.number_of_edges())
print("First original graph nodes:", list(G_original.nodes())[:10])

print("\n=== Original graph alignment: all-features ===")
check_graph_feature_alignment(
    G=G_original,
    features_df=features_df,
    features_tensor=features_tensor
)

print("\n=== Original graph alignment: ARD circles-only ===")
check_graph_feature_alignment(
    G=G_original,
    features_df=ard_df,
    features_tensor=ard_tensor
)

print("\n=== Original graph alignment: graph-structure-only identity ===")
check_graph_feature_alignment(
    G=G_original,
    features_df=structure_only_df,
    features_tensor=structure_only_tensor
)

In [ ]:
# ============================================================
# CELL 14 - Helper to run all DGI methods for one feature set
# ============================================================

def run_all_dgi_methods_for_feature_set(
    feature_set_name,
    feature_tensor,
    graphs,
    G_original,
    output_folder,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123
):
    print("\n" + "=" * 80)
    print(f"RUNNING DGI FOR FEATURE SET: {feature_set_name}")
    print("=" * 80)

    # --------------------------------------------------------
    # METHOD 1 - Independent ensemble
    # --------------------------------------------------------

    print("\nMETHOD 1 - Independent DGI ensemble")

    embeddings_method1, loss_histories_method1 = (
        train_dgi_ensemble_mean_independent(
            graphs=graphs,
            features_tensor=feature_tensor,
            embedding_dim=embedding_dim,
            lr=lr,
            max_epochs=max_epochs,
            patience=patience,
            seed=seed,
            verbose=False,
        )
    )

    save_embeddings(
        embeddings_method1,
        os.path.join(
            output_folder,
            f"embeddings_method1_independent_mean_{feature_set_name}.npy"
        )
    )

    save_loss_histories(
        loss_histories_method1,
        os.path.join(
            output_folder,
            f"dgi_loss_method1_independent_{feature_set_name}.pkl"
        )
    )

    print("Saved METHOD 1:", embeddings_method1.shape)

    # --------------------------------------------------------
    # METHOD 2 - Shared sequential
    # --------------------------------------------------------

    print("\nMETHOD 2 - Shared sequential DGI")

    (
        embeddings_method2,
        all_graph_embeddings_method2,
        loss_history_method2,
        model_method2,
        training_embeddings_method2,
        training_best_epoch_method2,
        training_best_loss_method2,
    ) = train_dgi_shared_across_graphs(
        graphs=graphs,
        features_tensor=feature_tensor,
        embedding_dim=embedding_dim,
        lr=lr,
        max_epochs=max_epochs,
        patience=patience,
        seed=seed,
        verbose=True,
    )

    # Current/post-training output
    save_embeddings(
        embeddings_method2,
        os.path.join(
            output_folder,
            f"embeddings_method2_shared_sequential_mean_{feature_set_name}.npy"
        )
    )

    # New training-time output
    save_embeddings(
        training_embeddings_method2,
        os.path.join(
            output_folder,
            f"embeddings_method2_shared_sequential_TRAINING_MEAN_"
            f"{feature_set_name}.npy"
        )
    )

    save_loss_histories(
        loss_history_method2,
        os.path.join(
            output_folder,
            f"dgi_loss_method2_shared_sequential_{feature_set_name}.pkl"
        )
    )

    save_model_state(
        model_method2,
        os.path.join(
            output_folder,
            f"dgi_model_method2_shared_sequential_{feature_set_name}.pt"
        )
    )

    print("Saved METHOD 2 post-training:", embeddings_method2.shape)
    print(
        "Saved METHOD 2 training-time:",
        training_embeddings_method2.shape
    )
    print(
        "METHOD 2 best training epoch/loss:",
        training_best_epoch_method2,
        training_best_loss_method2
    )

    # --------------------------------------------------------
    # METHOD 3 - Batch DGI
    # --------------------------------------------------------

    print("\nMETHOD 3 - Batch DGI")

    (
        embeddings_method3,
        all_graph_embeddings_method3,
        loss_history_method3,
        model_method3,
        training_embeddings_method3,
        training_best_epoch_method3,
        training_best_loss_method3,
    ) = train_dgi_batch_across_graphs(
        graphs=graphs,
        features_tensor=feature_tensor,
        embedding_dim=embedding_dim,
        lr=lr,
        max_epochs=max_epochs,
        patience=patience,
        seed=seed,
        batch_size=None,  # all reconstructed graphs in one batch
        verbose=True,
    )

    # Current/post-training output
    save_embeddings(
        embeddings_method3,
        os.path.join(
            output_folder,
            f"embeddings_method3_batch_mean_{feature_set_name}.npy"
        )
    )

    # New training-time output
    save_embeddings(
        training_embeddings_method3,
        os.path.join(
            output_folder,
            f"embeddings_method3_batch_TRAINING_MEAN_"
            f"{feature_set_name}.npy"
        )
    )

    save_loss_histories(
        loss_history_method3,
        os.path.join(
            output_folder,
            f"dgi_loss_method3_batch_{feature_set_name}.pkl"
        )
    )

    save_model_state(
        model_method3,
        os.path.join(
            output_folder,
            f"dgi_model_method3_batch_{feature_set_name}.pt"
        )
    )

    print("Saved METHOD 3 post-training:", embeddings_method3.shape)
    print(
        "Saved METHOD 3 training-time:",
        training_embeddings_method3.shape
    )
    print(
        "METHOD 3 best training epoch/loss:",
        training_best_epoch_method3,
        training_best_loss_method3
    )

    # --------------------------------------------------------
    # ORIGINAL GRAPH
    # --------------------------------------------------------

    print("\nORIGINAL GRAPH DGI")

    (
        embeddings_original,
        loss_history_original,
        training_embeddings_original,
        training_best_epoch_original,
        training_best_loss_original,
    ) = train_dgi_single_graph(
        G=G_original,
        features_tensor=feature_tensor,
        embedding_dim=embedding_dim,
        lr=lr,
        max_epochs=max_epochs,
        patience=patience,
        seed=seed,
        verbose=True,
    )

    # Current/post-training output
    save_embeddings(
        embeddings_original,
        os.path.join(
            output_folder,
            f"embeddings_original_{feature_set_name}.npy"
        )
    )

    # New training-time output for the original graph.
    # There is no averaging across graph realizations because
    # the original dataset contains only one graph.
    save_embeddings(
        training_embeddings_original,
        os.path.join(
            output_folder,
            f"embeddings_original_TRAINING_{feature_set_name}.npy"
        )
    )

    save_loss_histories(
        loss_history_original,
        os.path.join(
            output_folder,
            f"dgi_loss_original_{feature_set_name}.pkl"
        )
    )

    print("Saved ORIGINAL post-training:", embeddings_original.shape)
    print(
        "Saved ORIGINAL training-time:",
        training_embeddings_original.shape
    )
    print(
        "ORIGINAL best training epoch/loss:",
        training_best_epoch_original,
        training_best_loss_original
    )

    return {
        "method1": embeddings_method1,
        "method2": embeddings_method2,
        "method2_training": training_embeddings_method2,
        "method3": embeddings_method3,
        "method3_training": training_embeddings_method3,
        "original": embeddings_original,
        "original_training": training_embeddings_original,
    }


In [ ]:
# ============================================================
# CELL 15 - Run DGI with all features
# ============================================================

results_all_features = run_all_dgi_methods_for_feature_set(
    feature_set_name="all_features",
    feature_tensor=features_tensor,
    graphs=graphs,
    G_original=G_original,
    output_folder=output_folder,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123
)

In [ ]:
# ============================================================
# CELL 16 - Run DGI with ARD circles-only
# ============================================================

results_ard_circles_only = run_all_dgi_methods_for_feature_set(
    feature_set_name="ard_circles_only",
    feature_tensor=ard_tensor,
    graphs=graphs,
    G_original=G_original,
    output_folder=output_folder,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    seed=123
)

In [ ]:
# ============================================================
# CELL 17 - Run DGI with graph-structure-only identity matrix
# ============================================================

results_graph_structure_identity_only = (
    run_all_dgi_methods_for_feature_set(
        feature_set_name="graph_structure_identity_only",
        feature_tensor=structure_only_tensor,
        graphs=graphs,
        G_original=G_original,
        output_folder=output_folder,
        embedding_dim=64,
        lr=0.001,
        max_epochs=500,
        patience=20,
        seed=123
    )
)

In [ ]:
# ============================================================
# CELL 18 - Summary of saved embeddings
# ============================================================

print("Saved embeddings in:", output_folder)

for filename in sorted(os.listdir(output_folder)):
    if filename.endswith(".npy"):
        print(filename)

print("\nNew diagnostic outputs:")
print("- embeddings_original_TRAINING_<feature_set>.npy")
print(
    "- embeddings_method2_shared_sequential_TRAINING_MEAN_<feature_set>.npy"
)
print(
    "- embeddings_method3_batch_TRAINING_MEAN_<feature_set>.npy"
)
print(
    "\nMETHOD 1 remains unchanged because each reconstruction already "
    "uses its own independently trained encoder."
)
